# LLM Proof Benchmark — Google Colab workflow

Before starting, choose **Runtime → Change runtime type → T4 GPU**. Then run **1**, **2**, **3**, **3a**, and exactly one of **4a–4d**. For local models only, run **5a**, then **6–7**. For three local plus four manually supplied online proofs, run **5b**, **5c** to upload the proofs, then **5d** when you are ready to grade them, followed by **6–7**. Finally run **9–10** after ChatGPT has returned the evaluation JSON. Do not begin with cell 5: it needs the repository and packages installed by cells 1 and 3. Colab's `/content` folder is temporary, so download the final archive before the runtime ends.

## 1. Clone or update the project

**Outcome:** the repository is available at `/content/llm-proof-benchmark`, and the final line shows the installed Git commit. Re-running this cell updates an existing copy instead of cloning a second one.

In [ ]:
from pathlib import Path

%cd /content
if not Path('llm-proof-benchmark').exists():
    !git clone https://github.com/WaydenDunford/llm-proof-benchmark.git
%cd /content/llm-proof-benchmark
!git pull --ff-only
!git log -1 --oneline

## 2. Confirm the GPU

**Outcome:** the output should name a Tesla T4 and say `CUDA available: True`. If it does not, change the runtime type to T4 GPU and run this cell again.

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 3. Install benchmark dependencies

**Outcome:** the project is installed in editable mode, while Colab's GPU-enabled PyTorch remains in place. `bitsandbytes` enables 4-bit loading so the 7B model fits on the T4.

In [ ]:
%cd /content/llm-proof-benchmark
!pip install -q -e '.[dev]' transformers accelerate bitsandbytes

## 3a. Choose the theorem and proof length

**Where to put a theorem:** every theorem is a YAML file in the repository's `theorems/` folder. The supplied example is `theorems/theorem_001.yaml` with ID `T001`. To add your own, copy that file in the Colab file panel, edit its `id`, `title`, `statement`, `assumptions`, `allowed_context`, and `reference_proof`, then set `THEOREM_ID` below to the new ID.

`MAX_NEW_TOKENS` limits generated tokens, not characters. There is no notebook character limit for the saved proof. A 1,024-token limit is usually several thousand characters, depending on mathematical notation. The full proof is saved in JSON even if Colab shortens a displayed output.

In [ ]:
THEOREM_ID = 'T001'        # Change this to the id in your own YAML theorem file.
MAX_NEW_TOKENS = 1024      # 512 is shorter; 2048 allows a longer proof but takes longer.

!python -m src.cli list-theorems
print(f'Chosen theorem: {THEOREM_ID}; maximum generated tokens: {MAX_NEW_TOKENS}')

## 4a. DeepSeek-R1-Distill-Qwen-7B

**Outcome:** only DeepSeek-R1-Distill-Qwen-7B is enabled for this session. Run this cell, then run cell 5. Choose exactly one of 4a, 4b, 4c, or 4d before running cell 5.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/models.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
selected_model = 'deepseek-r1-distill-qwen-7b'
for model in config['models']:
    model['enabled'] = model['name'] == selected_model
    model['load_in_4bit'] = True
    model['device'] = 'auto'
    model['dtype'] = 'float16'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Enabled:', [m['name'] for m in config['models'] if m['enabled']])

## 4b. DeepTheorem-Qwen-7B-RL

**Outcome:** only DeepTheorem-Qwen-7B-RL is enabled for this session. Run this cell, then run cell 5. Choose exactly one of 4a, 4b, 4c, or 4d before running cell 5.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/models.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
selected_model = 'deeptheorem-qwen-7b'
for model in config['models']:
    model['enabled'] = model['name'] == selected_model
    model['load_in_4bit'] = True
    model['device'] = 'auto'
    model['dtype'] = 'float16'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Enabled:', [m['name'] for m in config['models'] if m['enabled']])

## 4c. Qwen3-8B

**Outcome:** only Qwen3-8B is enabled for this session. Run this cell, then run cell 5. Choose exactly one of 4a, 4b, 4c, or 4d before running cell 5.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/models.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
selected_model = 'qwen3'
for model in config['models']:
    model['enabled'] = model['name'] == selected_model
    model['load_in_4bit'] = True
    model['device'] = 'auto'
    model['dtype'] = 'float16'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Enabled:', [m['name'] for m in config['models'] if m['enabled']])

## 4d. All three models, sequentially

**Outcome:** all models are selected for one comparison run. They are not loaded together: the benchmark loads one 4-bit model, generates a proof, unloads it, clears GPU memory, and then loads the next. Including Math-Shepherd, the first all-model run downloads roughly 55–60 GB to temporary Colab storage and can take much longer. Run this cell, then run cell 5.

In [ ]:
import yaml
from pathlib import Path

config_path = Path('configs/models.yaml')
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
for model in config['models']:
    model['enabled'] = True
    model['load_in_4bit'] = True
    model['device'] = 'auto'
    model['dtype'] = 'float16'
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Sequential comparison:', [m['name'] for m in config['models'] if m['enabled']])

## 5a. Generate and evaluate local-model proofs only

**Outcome:** a real model-generated proof, Math-Shepherd step scores, canonical proofs/evaluations JSON files, an Excel workbook, and a blind Markdown bundle are created. For one selected model, the first run downloads roughly 30 GB of model files into the temporary Colab runtime: the generator plus the separate Math-Shepherd checkpoint. An all-model sequential run needs roughly 55–60 GB and may take substantially longer.

Math-Shepherd loads only after the generator has been unloaded. It uses the real checkpoint in 4-bit mode, but the benchmark adapts it to theorem proofs by scoring each numbered step from its `+` versus `-` next-token probabilities. These are useful auxiliary model scores, not a formal proof check or an official Math-Shepherd theorem-proof score.

In [ ]:
from pathlib import Path

repo = Path('/content/llm-proof-benchmark')
if not repo.is_dir():
    raise RuntimeError('Run cells 1–4 before cell 5. The repository is missing from /content.')
%cd /content/llm-proof-benchmark
!python -m src.cli prepare-dual-evaluation --theorem {THEOREM_ID} --prompt structured --max-new-tokens {MAX_NEW_TOKENS} --temperature 0 --no-sample --math-shepherd-backend transformers

## 5b. Generate local proofs for a combined local + online run

Use this path when you want all three local models plus manually generated OpenAI, Claude, DeepSeek, and Gemini proofs in one final comparison. First run **4d**, then run this cell. It generates the local proofs only; Math-Shepherd starts in cell 5d.

In [ ]:
%cd /content/llm-proof-benchmark
!python -m src.cli run --theorem {THEOREM_ID} --prompt structured --max-new-tokens {MAX_NEW_TOKENS} --temperature 0 --no-sample

## 5c. Upload four online proofs

Before running the code cell, create one Markdown file for each online model. Each file must use this exact format; replace the proof text after the second `---`:

```text
---
model_name: OpenAI
model_id: manual/openai
---
S1. First proof step.
S2. Second proof step.
```

Use distinct `model_name` and `model_id` values for the four online proofs. Upload all four `.md` files in the same picker. This cell validates them, adds them to the current run, and prints all seven proofs with a **READY FOR GRADING** confirmation. It does not start Math-Shepherd.

In [ ]:
from google.colab import files
from pathlib import Path
from src.dual import add_manual_proofs
import json

proof_files = sorted(Path('results/proofs').glob(f'{THEOREM_ID}_*_proofs.json'))
if not proof_files:
    raise RuntimeError('Run cell 5b first to generate the three local proofs.')
proofs_file = proof_files[-1]
run = json.loads(proofs_file.read_text(encoding='utf-8'))
local = [p for p in run['proofs'] if p['backend'] != 'manual']
if len(local) != 3 or any(p['status'] != 'success' or not p['proof'] for p in local):
    raise RuntimeError('Expected three successful local proofs. Check cell 5b before uploading.')

uploaded = files.upload()
if len(uploaded) != 4 or any(not name.lower().endswith('.md') for name in uploaded):
    raise ValueError('Select exactly four Markdown (.md) proof files together; nothing was added to the run.')
online_directory = Path('online_proofs') / run['run_id']
online_directory.mkdir(parents=True, exist_ok=True)
for old_file in online_directory.glob('*.md'):
    old_file.unlink()
for filename, content in uploaded.items():
    (online_directory / Path(filename).name).write_bytes(content)
add_manual_proofs(Path.cwd(), proofs_file, online_directory)
ready = json.loads(proofs_file.read_text(encoding='utf-8'))
if len(ready['proofs']) != 7 or any(p['status'] != 'success' or not p['proof_id'] for p in ready['proofs']):
    raise RuntimeError('The run does not contain seven successful proofs. Grading is blocked.')
print('READY FOR GRADING — run', ready['run_id'])
print('Proofs file:', proofs_file)
for proof in ready['proofs']:
    print(proof['proof_id'], '|', proof['model_name'], '|', proof['source_file'] if proof['backend'] == 'manual' else 'local model')

## 5d. Start grading the seven proofs

Run this cell only after cell 5c prints **READY FOR GRADING** and you have checked all seven proof names. It verifies the count again, then runs Math-Shepherd, updates Excel, and creates one anonymous ChatGPT bundle.

In [ ]:
from pathlib import Path
from src.dual import bundle, math_shepherd
import json

proofs_file = sorted(Path('results/proofs').glob(f'{THEOREM_ID}_*_proofs.json'))[-1]
proofs = json.loads(proofs_file.read_text(encoding='utf-8'))
if len(proofs['proofs']) != 7 or sum(p['backend'] == 'manual' for p in proofs['proofs']) != 4:
    raise RuntimeError('Expected three local and four uploaded proofs. Run cell 5c first.')
if any(p['status'] != 'success' or not p['proof_id'] for p in proofs['proofs']):
    raise RuntimeError('Every proof must be successful and have a proof ID before grading.')
evaluation_file = math_shepherd(Path.cwd(), proofs_file, 'transformers')
evaluations = json.loads(evaluation_file.read_text(encoding='utf-8'))['evaluations']
if len(evaluations) != 7 or any(e['math_shepherd']['status'] != 'success' for e in evaluations):
    raise RuntimeError('Math-Shepherd did not grade all seven proofs. Inspect the evaluations file: ' + str(evaluation_file))
bundle_file, _ = bundle(Path.cwd(), proofs_file)
print('GRADED', len(evaluations), 'proofs:', ', '.join(e['proof_id'] for e in evaluations))
print('Evaluations:', evaluation_file)
print('ChatGPT bundle:', bundle_file)

## 6. Inspect the latest proof without flooding the notebook output

**Outcome:** you see the latest run ID, its model status, and an 800-character proof preview. The full proof remains in the JSON file.

In [ ]:
import json
from pathlib import Path

proofs_file = sorted(Path('results/proofs').glob(f'{THEOREM_ID}_*_proofs.json'))[-1]
proofs = json.loads(proofs_file.read_text(encoding='utf-8'))
print('Proofs file:', proofs_file)
print('Run ID:', proofs['run_id'])
for proof in proofs['proofs']:
    print(f"\n{proof['model_name']}: {proof['status']}")
    print(proof.get('proof_text', '')[:800])

## 7. Download the anonymous ChatGPT evaluation bundle

**Outcome:** your browser downloads one Markdown file. Upload this file to ChatGPT. Do not upload files from `results/evaluation_maps`, because they identify the model behind each anonymous proof.

In [ ]:
from google.colab import files
from pathlib import Path

bundle_file = sorted(Path('results/evaluation_input').glob(f'{THEOREM_ID}_*_chatgpt_bundle.md'))[-1]
print('Downloading:', bundle_file)
files.download(str(bundle_file))

## 8. Evaluate the bundle in ChatGPT

Upload the downloaded Markdown bundle to ChatGPT and follow its instruction to return **only the required JSON**. Save that JSON response as a file on your computer. Keep the proof IDs unchanged.

## 9. Upload and import ChatGPT's JSON evaluation

**Outcome:** the JSON evaluation is validated and merged into the same canonical evaluations file. The Excel workbook and reports are updated. This cell needs the same run's proof and Math-Shepherd evaluation files in `/content/llm-proof-benchmark/results`; if Colab reset, restore those files before importing. Run this only after completing the ChatGPT evaluation.

In [ ]:
from google.colab import files
from pathlib import Path
import json
import os
import sys

repo = Path('/content/llm-proof-benchmark')
if not (repo / 'src/cli.py').is_file():
    raise RuntimeError('Project files are missing. Run cells 1 and 3, then restore the results for this run if Colab reset.')
os.chdir(repo)
sys.path.insert(0, str(repo))
from src.dual import create_mapping, import_chatgpt

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one ChatGPT evaluation JSON file.')
evaluation_file = repo / next(iter(uploaded))
evaluation = json.loads(evaluation_file.read_text(encoding='utf-8-sig'))
run_id = evaluation['run_id']
theorem_id = evaluation['theorem_id']
proofs_file = repo / 'results/proofs' / f'{theorem_id}_{run_id}_proofs.json'
evaluations_file = repo / 'results/evaluations' / f'{theorem_id}_{run_id}_evaluations.json'
if not proofs_file.is_file() or not evaluations_file.is_file():
    raise RuntimeError('Matching proof and Math-Shepherd evaluation files are missing for run ' + run_id + '. Restore the results for that run before importing; a new run will have a different ID.')
mapping_file = repo / 'results/evaluation_maps' / f'{theorem_id}_{run_id}_mapping.json'
if not mapping_file.is_file():
    create_mapping(repo, proofs_file)
print('Importing evaluation for run:', run_id)
target = import_chatgpt(repo, run_id, evaluation_file)
print('Imported', len(json.loads(target.read_text(encoding='utf-8'))['evaluations']), 'proof evaluations into', target)

## 10. Download the finished results

**Outcome:** your browser downloads a zip archive containing the run's proofs, evaluations, reports, and Excel workbook. This keeps the results after Colab clears `/content`.

In [ ]:
from google.colab import files

!zip -r /content/llm-proof-benchmark-results.zip results/proofs results/evaluations results/evaluation_maps results/evaluation_input results/reports results/benchmark_results.xlsx
files.download('/content/llm-proof-benchmark-results.zip')

## Optional: copy results to Google Drive

**Outcome:** the finished zip is copied to `MyDrive/llm-proof-benchmark`. Run this after the download cell if you also want a Drive backup.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
drive_folder = Path('/content/drive/MyDrive/llm-proof-benchmark')
drive_folder.mkdir(parents=True, exist_ok=True)
shutil.copy2('/content/llm-proof-benchmark-results.zip', drive_folder / 'llm-proof-benchmark-results.zip')
print('Saved:', drive_folder / 'llm-proof-benchmark-results.zip')